In [1]:
import time
import csv
import re
from datetime import datetime, timedelta
from urllib.parse import quote, urlparse, parse_qs

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException

# ---------------------------------------------------------
# 설정
# ---------------------------------------------------------
KEYWORDS = ["1회 사용", "한번 사용", "잠깐 사용", "1회", "거의 새상품"]
REGIONS = ["문정동", "천호동", "잠원동", "신림동", "구로동", "화곡동",
           "상암동", "역촌동", "도봉동", "상계동", "상봉동"]

START_DATE = datetime(2025, 6, 1)
END_DATE = datetime(2026, 6, 21, 23, 59, 59)
TODAY = datetime(2026, 6, 16)          # 상대시간 환산 기준일
OUTPUT_FILE = "daangn_crawl_by_region_kw2.csv"

SCHEMA = ["post_id", "platform", "keyword", "title", "content_text",
          "view_count", "comment_count", "posted_at", "region"]

SCROLL_TIMES = 30
PAGE_WAIT = 3.0

# 지역코드를 못 찾았을 때를 대비한 수동 매핑(알면 채워두면 됨, 비워도 자동탐색함)
# 예: "문정동": "문정동-6184"
REGION_CODE_MAP = {
    "문정동": "문정동-6184",   # 사용자가 알려준 값
}

# ---------------------------------------------------------
# 드라이버
# ---------------------------------------------------------
options = Options()
# options.add_argument("--headless=new")
options.add_argument("--window-size=1400,1000")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                     "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36")
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)


# ---------------------------------------------------------
# 유틸
# ---------------------------------------------------------
def to_int(text):
    if not text:
        return None
    nums = re.sub(r"[^\d]", "", text)
    return int(nums) if nums else None


def extract_post_id(url):
    m = re.search(r"-([a-z0-9]{8,})/?$", url)
    if m:
        slug = m.group(1)
        return int.from_bytes(slug.encode(), "big") % (10**18)
    return None


def parse_relative_or_abs(text):
    if not text:
        return None
    text = text.strip()
    for fmt in ("%Y-%m-%dT%H:%M:%S", "%Y-%m-%d %H:%M:%S",
                "%Y.%m.%d", "%Y-%m-%d", "%Y. %m. %d"):
        try:
            return datetime.strptime(text[:len(fmt) + 4].strip(), fmt)
        except ValueError:
            continue
    t = text.replace("끌올", "").strip()
    if "방금" in t or "분 전" in t or "초 전" in t or "시간 전" in t:
        return TODAY
    m = re.search(r"(\d+)\s*일\s*전", t)
    if m:
        return TODAY - timedelta(days=int(m.group(1)))
    m = re.search(r"(\d+)\s*주\s*전", t)
    if m:
        return TODAY - timedelta(weeks=int(m.group(1)))
    m = re.search(r"(\d+)\s*개월\s*전", t)
    if m:
        return TODAY - timedelta(days=int(m.group(1)) * 30)
    m = re.search(r"(\d+)\s*년\s*전", t)
    if m:
        return TODAY - timedelta(days=int(m.group(1)) * 365)
    return None


def in_period(dt):
    return dt is not None and START_DATE <= dt <= END_DATE


def safe_text(by, sel, root=None):
    try:
        return (root or driver).find_element(by, sel).text.strip()
    except Exception:
        return None


def safe_attr(by, sel, attr, root=None):
    try:
        return (root or driver).find_element(by, sel).get_attribute(attr)
    except Exception:
        return None


# ---------------------------------------------------------
# 지역코드 자동 탐색
# ---------------------------------------------------------
def find_region_code(region):
    if region in REGION_CODE_MAP and REGION_CODE_MAP[region]:
        return REGION_CODE_MAP[region]

    probe = f"https://www.daangn.com/kr/buy-sell/?search={quote(region)}"
    driver.get(probe)
    time.sleep(PAGE_WAIT)

    qs = parse_qs(urlparse(driver.current_url).query)
    if "in" in qs and qs["in"]:
        return qs["in"][0]

    for a in driver.find_elements(By.CSS_SELECTOR, "a[href*='in=']"):
        href = a.get_attribute("href") or ""
        m = re.search(r"in=([^&]+)", href)
        if m:
            code = m.group(1)
            from urllib.parse import unquote
            if region in unquote(code):
                return unquote(code)
    return None


# ---------------------------------------------------------
# 검색결과 → 게시물 링크 수집 (지역코드 포함)
# ---------------------------------------------------------
def collect_links(keyword, region_code):
    url = (f"https://www.daangn.com/kr/buy-sell/s/"
           f"?in={quote(region_code)}&search={quote(keyword)}")
    driver.get(url)
    time.sleep(PAGE_WAIT)

    last_h = driver.execute_script("return document.body.scrollHeight")
    for _ in range(SCROLL_TIMES):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1.8)
        new_h = driver.execute_script("return document.body.scrollHeight")
        if new_h == last_h:
            break
        last_h = new_h

    links, seen = [], set()
    for a in driver.find_elements(By.CSS_SELECTOR, "a[href*='/kr/buy-sell/']"):
        href = a.get_attribute("href")
        if href and re.search(r"/kr/buy-sell/.+-[a-z0-9]{8,}/?$", href):
            if href not in seen:
                seen.add(href)
                links.append(href)
    return links


# ---------------------------------------------------------
# 상세 페이지 크롤링
# ---------------------------------------------------------
def crawl_detail(url, keyword, region):
    driver.get(url)
    time.sleep(PAGE_WAIT)

    post_id = extract_post_id(url)

    title = (safe_text(By.CSS_SELECTOR, "h1")
             or safe_attr(By.CSS_SELECTOR, "meta[property='og:title']", "content"))

    content_text = (safe_text(By.CSS_SELECTOR, "[data-gtm='article_description']")
                    or safe_text(By.CSS_SELECTOR, "article")
                    or safe_attr(By.CSS_SELECTOR, "meta[property='og:description']", "content"))

    dt = (parse_relative_or_abs(safe_attr(By.CSS_SELECTOR, "time", "datetime"))
          or parse_relative_or_abs(safe_text(By.CSS_SELECTOR, "time")))
    if dt is None:
        body = safe_text(By.TAG_NAME, "body") or ""
        m = re.search(r"(끌올\s*)?\d+\s*(일|주|개월|년)\s*전|방금\s*전", body)
        if m:
            dt = parse_relative_or_abs(m.group(0))

    body_text = safe_text(By.TAG_NAME, "body") or ""
    vm = re.search(r"조회\s*([\d,]+)", body_text)
    cm = re.search(r"(?:댓글|관심|채팅)\s*([\d,]+)", body_text)
    view_count = to_int(vm.group(1)) if vm else None
    comment_count = to_int(cm.group(1)) if cm else None

    return {
        "post_id": post_id,
        "platform": "carrot",
        "keyword": keyword,
        "title": title,
        "content_text": content_text,
        "view_count": view_count,
        "comment_count": comment_count,
        "posted_at": dt.strftime("%Y-%m-%d %H:%M:%S") if dt else None,
        "region": region,
    }, dt


# ---------------------------------------------------------
# 메인: 지역 × 키워드
# ---------------------------------------------------------
results = []
seen_ids = set()

try:
    for region in REGIONS:
        code = find_region_code(region)
        if not code:
            print(f"\n##### [{region}] 지역코드 탐색 실패 → 건너뜀 "
                  f"(REGION_CODE_MAP에 수동 입력 권장) #####")
            continue
        print(f"\n##### 지역: {region}  (코드: {code}) #####")

        for kw in KEYWORDS:
            print(f"  ===== 키워드: {kw} =====")
            links = collect_links(kw, code)
            print(f"  수집된 게시물 링크: {len(links)}")

            for i, url in enumerate(links, 1):
                try:
                    row, dt = crawl_detail(url, kw, region)

                    if row["post_id"] in seen_ids:
                        continue
                    if not in_period(dt):
                        continue

                    seen_ids.add(row["post_id"])
                    results.append(row)
                    print(f"    [{i}/{len(links)}] 수집: {row['title']} ({row['posted_at']})")

                except (StaleElementReferenceException, TimeoutException) as e:
                    print(f"    [{i}/{len(links)}] {type(e).__name__} skip")
                except Exception as e:
                    print(f"    [{i}/{len(links)}] 에러: {e}")
                time.sleep(1)

finally:
    driver.quit()

# ---------------------------------------------------------
# CSV 저장
# ---------------------------------------------------------
with open(OUTPUT_FILE, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=SCHEMA)
    writer.writeheader()
    writer.writerows(results)

print(f"\n완료: 총 {len(results)}건 저장 → {OUTPUT_FILE}")


##### 지역: 문정동  (코드: 문정동-6184) #####
  ===== 키워드: 1회 사용 =====
  수집된 게시물 링크: 60
    [1/60] 수집: narka 헤어 세범 미스트 (2026-06-14 00:00:00)
    [2/60] 수집: 일투 리쿠 타올 슬로건 (2026-06-07 00:00:00)
    [3/60] 수집: 볼리움 고주파 뷰티디바이스 실버 (2026-06-14 00:00:00)
    [4/60] 수집: 용돈 ATM 팝니다. (2026-06-16 00:00:00)
    [5/60] 수집: 보드게임 쿼리도 (2026-06-15 00:00:00)
    [6/60] 수집: 다이소 미니 헤어드라이기 (2026-06-06 00:00:00)
    [7/60] 수집: 파라핀베스 bh-300 편한민족 국산 (2026-06-12 00:00:00)
    [8/60] 수집: 삼성전자 세라믹 전자레인지 다이얼식 23L(상태 S급) (2026-06-09 00:00:00)
    [9/60] 수집: 캠핑 원액션 테이블 (SOTO 필드호퍼) (2026-06-14 00:00:00)
    [10/60] 수집: 히카리코류 505 블런트가위 (2026-06-13 00:00:00)
    [11/60] 수집: 필립스 이지터치 플러스 스팀다리미 (2026-06-11 00:00:00)
    [12/60] 수집: 결이고은 문정역점 “작은얼굴관리“ 16회 양도 (2026-06-01 00:00:00)
    [13/60] 수집: (1회 사용) 제니퍼룸 무선 차량 휴대용 미니 핸디청소기 (2026-06-16 00:00:00)
    [14/60] 수집: 제이드 롤러 (2026-06-05 00:00:00)
    [15/60] 수집: 네일 국시 재료 (모스티브 팁 커터+글루+젤글루) (2026-06-14 00:00:00)
    [16/60] 수집: 포렌코즈 타투 끌레르 벨벳 틴트 - 핑크러쉬 (2026-06-13 00:00:00)
    [17/60] 

InvalidSessionIdException: Message: invalid session id; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff6c9547de5+14895]
	chromedriver!GetHandleVerifier [0x7ff6c9547e50+14900]
	chromedriver!(No symbol) [0x7ff6c92ad396]
	chromedriver!(No symbol) [0x7ff6c92f8e76]
	chromedriver!(No symbol) [0x7ff6c932ff62]
	chromedriver!(No symbol) [0x7ff6c9329d7f]
	chromedriver!(No symbol) [0x7ff6c9329543]
	chromedriver!(No symbol) [0x7ff6c9276325]
	chromedriver!GetHandleVerifier [0x7ff6c985cc49+3296f9]
	chromedriver!GetHandleVerifier [0x7ff6c9857375+323e25]
	chromedriver!GetHandleVerifier [0x7ff6c987bc82+348732]
	chromedriver!GetHandleVerifier [0x7ff6c9566045+32af5]
	chromedriver!GetHandleVerifier [0x7ff6c956ecec+3b79c]
	chromedriver!(No symbol) [0x7ff6c92750bc]
	chromedriver!GetHandleVerifier [0x7ff6c99c0b18+48d5c8]
	KERNEL32!BaseThreadInitThunk [0x7ffb22487

In [ ]:
import time
import csv
import os
import re
from datetime import datetime, timedelta
from urllib.parse import quote

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException

# ---------------------------------------------------------
# 설정
# ---------------------------------------------------------
KEYWORDS = ["처분", "무료나눔", "양도"]

# 지역코드 하드코딩 (테스트: 4개 지역)
REGION_CODE_MAP = {
    "문정동": "문정동-6184",
    "천호동": "천호동-6044",
    "잠원동": "잠원동-367",
    "신림동": "신림동-355",
}

START_DATE = datetime(2025, 6, 1)
END_DATE = datetime(2026, 6, 21, 23, 59, 59)
TODAY = datetime(2026, 6, 16)
OUTPUT_FILE = "daangn_crawl_test4.csv"

SCHEMA = ["post_id", "platform", "keyword", "title", "content_text",
          "view_count", "comment_count", "posted_at", "region"]

SCROLL_TIMES = 30
PAGE_WAIT = 3.0
RESTART_EVERY = 60        # 게시물 N건 처리할 때마다 브라우저 재시작(메모리 누수 방지)


# ---------------------------------------------------------
# 드라이버 생성/재시작
# ---------------------------------------------------------
def make_driver():
    options = Options()
    # options.add_argument("--headless=new")
    options.add_argument("--window-size=1400,1000")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--disable-dev-shm-usage")   # 메모리 부족(Out of Memory) 완화
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-gpu")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                         "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36")
    return webdriver.Chrome(options=options)


driver = make_driver()


# ---------------------------------------------------------
# 유틸
# ---------------------------------------------------------
def to_int(text):
    if not text:
        return None
    nums = re.sub(r"[^\d]", "", text)
    return int(nums) if nums else None


def extract_post_id(url):
    m = re.search(r"-([a-z0-9]{8,})/?$", url)
    if m:
        slug = m.group(1)
        return int.from_bytes(slug.encode(), "big") % (10**18)
    return None


def parse_relative_or_abs(text):
    if not text:
        return None
    text = text.strip()
    for fmt in ("%Y-%m-%dT%H:%M:%S", "%Y-%m-%d %H:%M:%S",
                "%Y.%m.%d", "%Y-%m-%d", "%Y. %m. %d"):
        try:
            return datetime.strptime(text[:len(fmt) + 4].strip(), fmt)
        except ValueError:
            continue
    t = text.replace("끌올", "").strip()
    if "방금" in t or "분 전" in t or "초 전" in t or "시간 전" in t:
        return TODAY
    m = re.search(r"(\d+)\s*일\s*전", t)
    if m:
        return TODAY - timedelta(days=int(m.group(1)))
    m = re.search(r"(\d+)\s*주\s*전", t)
    if m:
        return TODAY - timedelta(weeks=int(m.group(1)))
    m = re.search(r"(\d+)\s*개월\s*전", t)
    if m:
        return TODAY - timedelta(days=int(m.group(1)) * 30)
    m = re.search(r"(\d+)\s*년\s*전", t)
    if m:
        return TODAY - timedelta(days=int(m.group(1)) * 365)
    return None


def in_period(dt):
    return dt is not None and START_DATE <= dt <= END_DATE


def safe_text(by, sel, root=None):
    try:
        return (root or driver).find_element(by, sel).text.strip()
    except Exception:
        return None


def safe_attr(by, sel, attr, root=None):
    try:
        return (root or driver).find_element(by, sel).get_attribute(attr)
    except Exception:
        return None


# ---------------------------------------------------------
# CSV 준비 (헤더 1회 작성, 이후 append)
# ---------------------------------------------------------
def init_csv():
    if not os.path.exists(OUTPUT_FILE):
        with open(OUTPUT_FILE, "w", encoding="utf-8-sig", newline="") as f:
            csv.DictWriter(f, fieldnames=SCHEMA).writeheader()


def append_row(row):
    with open(OUTPUT_FILE, "a", encoding="utf-8-sig", newline="") as f:
        csv.DictWriter(f, fieldnames=SCHEMA).writerow(row)


# ---------------------------------------------------------
# 검색결과 → 게시물 링크 수집
# ---------------------------------------------------------
def collect_links(keyword, region_code):
    url = (f"https://www.daangn.com/kr/buy-sell/s/"
           f"?in={quote(region_code)}&search={quote(keyword)}")
    driver.get(url)
    time.sleep(PAGE_WAIT)

    last_h = driver.execute_script("return document.body.scrollHeight")
    for _ in range(SCROLL_TIMES):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1.8)
        new_h = driver.execute_script("return document.body.scrollHeight")
        if new_h == last_h:
            break
        last_h = new_h

    links, seen = [], set()
    for a in driver.find_elements(By.CSS_SELECTOR, "a[href*='/kr/buy-sell/']"):
        href = a.get_attribute("href")
        if href and re.search(r"/kr/buy-sell/.+-[a-z0-9]{8,}/?$", href):
            if href not in seen:
                seen.add(href)
                links.append(href)
    return links


# ---------------------------------------------------------
# 상세 페이지 크롤링
# ---------------------------------------------------------
def crawl_detail(url, keyword, region):
    driver.get(url)
    time.sleep(PAGE_WAIT)

    post_id = extract_post_id(url)

    title = (safe_text(By.CSS_SELECTOR, "h1")
             or safe_attr(By.CSS_SELECTOR, "meta[property='og:title']", "content"))

    content_text = (safe_text(By.CSS_SELECTOR, "[data-gtm='article_description']")
                    or safe_text(By.CSS_SELECTOR, "article")
                    or safe_attr(By.CSS_SELECTOR, "meta[property='og:description']", "content"))

    dt = (parse_relative_or_abs(safe_attr(By.CSS_SELECTOR, "time", "datetime"))
          or parse_relative_or_abs(safe_text(By.CSS_SELECTOR, "time")))
    if dt is None:
        body = safe_text(By.TAG_NAME, "body") or ""
        m = re.search(r"(끌올\s*)?\d+\s*(일|주|개월|년)\s*전|방금\s*전", body)
        if m:
            dt = parse_relative_or_abs(m.group(0))

    body_text = safe_text(By.TAG_NAME, "body") or ""
    vm = re.search(r"조회\s*([\d,]+)", body_text)
    cm = re.search(r"(?:댓글|관심|채팅)\s*([\d,]+)", body_text)
    view_count = to_int(vm.group(1)) if vm else None
    comment_count = to_int(cm.group(1)) if cm else None

    return {
        "post_id": post_id,
        "platform": "carrot",
        "keyword": keyword,
        "title": title,
        "content_text": content_text,
        "view_count": view_count,
        "comment_count": comment_count,
        "posted_at": dt.strftime("%Y-%m-%d %H:%M:%S") if dt else None,
        "region": region,
    }, dt


# ---------------------------------------------------------
# 메인: 지역 × 키워드
# ---------------------------------------------------------
init_csv()
seen_ids = set()
total = 0
processed = 0      # 재시작 카운터

try:
    for region, code in REGION_CODE_MAP.items():
        print(f"\n##### 지역: {region}  (코드: {code}) #####")

        for kw in KEYWORDS:
            print(f"  ===== 키워드: {kw} =====")
            try:
                links = collect_links(kw, code)
            except Exception as e:
                print(f"  링크 수집 에러: {e} → 드라이버 재시작")
                try:
                    driver.quit()
                except Exception:
                    pass
                driver = make_driver()
                continue
            print(f"  수집된 게시물 링크: {len(links)}")

            for i, url in enumerate(links, 1):
                try:
                    row, dt = crawl_detail(url, kw, region)

                    if row["post_id"] in seen_ids:
                        continue
                    if not in_period(dt):
                        continue

                    seen_ids.add(row["post_id"])
                    append_row(row)          # 즉시 저장
                    total += 1
                    print(f"    [{i}/{len(links)}] 수집: {row['title']} ({row['posted_at']})")

                except (StaleElementReferenceException, TimeoutException) as e:
                    print(f"    [{i}/{len(links)}] {type(e).__name__} skip")
                except Exception as e:
                    print(f"    [{i}/{len(links)}] 에러: {e}")

                processed += 1
                if processed % RESTART_EVERY == 0:
                    print(f"    --- 드라이버 재시작 (누적 {processed}건 처리) ---")
                    try:
                        driver.quit()
                    except Exception:
                        pass
                    driver = make_driver()

                time.sleep(1)

finally:
    try:
        driver.quit()
    except Exception:
        pass

print(f"\n완료: 총 {total}건 저장 → {OUTPUT_FILE}")


##### 지역: 문정동  (코드: 문정동-6184) #####
  ===== 키워드: 처분 =====
  수집된 게시물 링크: 60
    [1/60] 수집: 옷장정리 : 🤍 (2026-06-16 00:00:00)
    [2/60] 수집: 포유문 출산준비물 (2026-06-16 00:00:00)
    [3/60] 수집: 한샘 시스템장 5개조 (2026-06-16 00:00:00)
    [4/60] 수집: 위시탈덕처분 (2026-06-09 00:00:00)
    [5/60] 수집: 산리오 포차코 소품 일괄 싸게처분✨ (2026-06-15 00:00:00)
    [6/60] 수집: [새상품] 딘토 브로우 픽서 (2026-06-16 00:00:00)
    [7/60] 수집: 태그호이어 까레라 칼리버 6 실버 오토매틱 시계 (2026-06-16 00:00:00)
    [8/60] 수집: [새상품] 앰플엔 블레미샷 잡티 크림 30ml (2026-06-16 00:00:00)
    [9/60] 수집: 티비장 중고 처분합니다 (2026-04-17 00:00:00)
    [10/60] 수집: [마지막가격][품절템] 몽슈레 루나크립 아기침대 풀세트 (2026-06-16 00:00:00)
    [11/60] 수집: 쓰리쿼터 데님 여름 반바지 팬츠L (2026-06-16 00:00:00)
    [12/60] 수집: >>새상품_아이보리 레이스 원피스 S사이즈 (2026-06-16 00:00:00)
    [13/60] 수집: 가스트로플러스 에어리움 40L 오븐에어프라이어 (6/30일 이후 처분) (2026-06-14 00:00:00)
    [14/60] 수집: 사카모토,프리렌,나루토,짱구,미쿠 라부부 처분중 (2026-06-13 00:00:00)
    [15/60] 수집: 보이넥스트도어 탈덕 처분 (2026-03-18 00:00:00)
    [16/60] 수집: 세라 saera구두 225cm 상태굿 처분 (2025-06-16 00:00:00)
    [1

In [1]:
import time
import csv
import os
import re
from datetime import datetime, timedelta
from urllib.parse import quote

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException

# ---------------------------------------------------------
# 설정
# ---------------------------------------------------------
KEYWORDS = ["처분", "무료나눔"]

# 지역코드 하드코딩 (테스트: 4개 지역)
REGION_CODE_MAP = {
    "문정동": "문정동-6184",
    "천호동": "천호동-6044",
}

START_DATE = datetime(2025, 6, 1)
END_DATE = datetime(2026, 6, 21, 23, 59, 59)
TODAY = datetime(2026, 6, 16)
OUTPUT_FILE = "daangn_crawl_test4.csv"

SCHEMA = ["post_id", "platform", "keyword", "title", "content_text",
          "view_count", "comment_count", "posted_at", "region"]

SCROLL_TIMES = 30
PAGE_WAIT = 3.0
RESTART_EVERY = 60        # 게시물 N건 처리할 때마다 브라우저 재시작(메모리 누수 방지)


# ---------------------------------------------------------
# 드라이버 생성/재시작
# ---------------------------------------------------------
def make_driver():
    options = Options()
    # options.add_argument("--headless=new")
    options.add_argument("--window-size=1400,1000")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--disable-dev-shm-usage")   # 메모리 부족(Out of Memory) 완화
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-gpu")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                         "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36")
    return webdriver.Chrome(options=options)


driver = make_driver()


# ---------------------------------------------------------
# 유틸
# ---------------------------------------------------------
def to_int(text):
    if not text:
        return None
    nums = re.sub(r"[^\d]", "", text)
    return int(nums) if nums else None


def extract_post_id(url):
    m = re.search(r"-([a-z0-9]{8,})/?$", url)
    if m:
        slug = m.group(1)
        return int.from_bytes(slug.encode(), "big") % (10**18)
    return None


def parse_relative_or_abs(text):
    if not text:
        return None
    text = text.strip()
    for fmt in ("%Y-%m-%dT%H:%M:%S", "%Y-%m-%d %H:%M:%S",
                "%Y.%m.%d", "%Y-%m-%d", "%Y. %m. %d"):
        try:
            return datetime.strptime(text[:len(fmt) + 4].strip(), fmt)
        except ValueError:
            continue
    t = text.replace("끌올", "").strip()
    if "방금" in t or "분 전" in t or "초 전" in t or "시간 전" in t:
        return TODAY
    m = re.search(r"(\d+)\s*일\s*전", t)
    if m:
        return TODAY - timedelta(days=int(m.group(1)))
    m = re.search(r"(\d+)\s*주\s*전", t)
    if m:
        return TODAY - timedelta(weeks=int(m.group(1)))
    m = re.search(r"(\d+)\s*개월\s*전", t)
    if m:
        return TODAY - timedelta(days=int(m.group(1)) * 30)
    m = re.search(r"(\d+)\s*년\s*전", t)
    if m:
        return TODAY - timedelta(days=int(m.group(1)) * 365)
    return None


def in_period(dt):
    return dt is not None and START_DATE <= dt <= END_DATE


def safe_text(by, sel, root=None):
    try:
        return (root or driver).find_element(by, sel).text.strip()
    except Exception:
        return None


def safe_attr(by, sel, attr, root=None):
    try:
        return (root or driver).find_element(by, sel).get_attribute(attr)
    except Exception:
        return None


# ---------------------------------------------------------
# CSV 준비 (헤더 1회 작성, 이후 append)
# ---------------------------------------------------------
def init_csv():
    if not os.path.exists(OUTPUT_FILE):
        with open(OUTPUT_FILE, "w", encoding="utf-8-sig", newline="") as f:
            csv.DictWriter(f, fieldnames=SCHEMA).writeheader()


def append_row(row):
    with open(OUTPUT_FILE, "a", encoding="utf-8-sig", newline="") as f:
        csv.DictWriter(f, fieldnames=SCHEMA).writerow(row)


# ---------------------------------------------------------
# 검색결과 → 게시물 링크 수집
# ---------------------------------------------------------
def collect_links(keyword, region_code):
    url = (f"https://www.daangn.com/kr/buy-sell/s/"
           f"?in={quote(region_code)}&search={quote(keyword)}")
    driver.get(url)
    time.sleep(PAGE_WAIT)

    last_h = driver.execute_script("return document.body.scrollHeight")
    for _ in range(SCROLL_TIMES):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1.8)
        new_h = driver.execute_script("return document.body.scrollHeight")
        if new_h == last_h:
            break
        last_h = new_h

    links, seen = [], set()
    for a in driver.find_elements(By.CSS_SELECTOR, "a[href*='/kr/buy-sell/']"):
        href = a.get_attribute("href")
        if href and re.search(r"/kr/buy-sell/.+-[a-z0-9]{8,}/?$", href):
            if href not in seen:
                seen.add(href)
                links.append(href)
    return links


# ---------------------------------------------------------
# 상세 페이지 크롤링
# ---------------------------------------------------------
def crawl_detail(url, keyword, region):
    driver.get(url)
    time.sleep(PAGE_WAIT)

    post_id = extract_post_id(url)

    title = (safe_text(By.CSS_SELECTOR, "h1")
             or safe_attr(By.CSS_SELECTOR, "meta[property='og:title']", "content"))

    content_text = (safe_text(By.CSS_SELECTOR, "[data-gtm='article_description']")
                    or safe_text(By.CSS_SELECTOR, "article")
                    or safe_attr(By.CSS_SELECTOR, "meta[property='og:description']", "content"))

    dt = (parse_relative_or_abs(safe_attr(By.CSS_SELECTOR, "time", "datetime"))
          or parse_relative_or_abs(safe_text(By.CSS_SELECTOR, "time")))
    if dt is None:
        body = safe_text(By.TAG_NAME, "body") or ""
        m = re.search(r"(끌올\s*)?\d+\s*(일|주|개월|년)\s*전|방금\s*전", body)
        if m:
            dt = parse_relative_or_abs(m.group(0))

    body_text = safe_text(By.TAG_NAME, "body") or ""
    vm = re.search(r"조회\s*([\d,]+)", body_text)
    cm = re.search(r"(?:댓글|관심|채팅)\s*([\d,]+)", body_text)
    view_count = to_int(vm.group(1)) if vm else None
    comment_count = to_int(cm.group(1)) if cm else None

    return {
        "post_id": post_id,
        "platform": "carrot",
        "keyword": keyword,
        "title": title,
        "content_text": content_text,
        "view_count": view_count,
        "comment_count": comment_count,
        "posted_at": dt.strftime("%Y-%m-%d %H:%M:%S") if dt else None,
        "region": region,
    }, dt


# ---------------------------------------------------------
# 메인: 지역 × 키워드
# ---------------------------------------------------------
init_csv()
seen_ids = set()
total = 0
processed = 0      # 재시작 카운터

try:
    for region, code in REGION_CODE_MAP.items():
        print(f"\n##### 지역: {region}  (코드: {code}) #####")

        for kw in KEYWORDS:
            print(f"  ===== 키워드: {kw} =====")
            try:
                links = collect_links(kw, code)
            except Exception as e:
                print(f"  링크 수집 에러: {e} → 드라이버 재시작")
                try:
                    driver.quit()
                except Exception:
                    pass
                driver = make_driver()
                continue
            print(f"  수집된 게시물 링크: {len(links)}")

            for i, url in enumerate(links, 1):
                try:
                    row, dt = crawl_detail(url, kw, region)

                    if row["post_id"] in seen_ids:
                        continue
                    if not in_period(dt):
                        continue

                    seen_ids.add(row["post_id"])
                    append_row(row)          # 즉시 저장
                    total += 1
                    print(f"    [{i}/{len(links)}] 수집: {row['title']} ({row['posted_at']})")

                except (StaleElementReferenceException, TimeoutException) as e:
                    print(f"    [{i}/{len(links)}] {type(e).__name__} skip")
                except Exception as e:
                    print(f"    [{i}/{len(links)}] 에러: {e}")

                processed += 1
                if processed % RESTART_EVERY == 0:
                    print(f"    --- 드라이버 재시작 (누적 {processed}건 처리) ---")
                    try:
                        driver.quit()
                    except Exception:
                        pass
                    driver = make_driver()

                time.sleep(1)

finally:
    try:
        driver.quit()
    except Exception:
        pass

print(f"\n완료: 총 {total}건 저장 → {OUTPUT_FILE}")


##### 지역: 문정동  (코드: 문정동-6184) #####
  ===== 키워드: 처분 =====
  수집된 게시물 링크: 60
    [1/60] 수집: 포유문 출산준비물 (2026-06-16 00:00:00)
    [2/60] 수집: 위시탈덕처분 (2026-06-09 00:00:00)
    [3/60] 수집: 한샘 시스템장 5개조 (2026-06-16 00:00:00)
    [4/60] 수집: 산리오 포차코 소품 일괄 싸게처분✨ (2026-06-15 00:00:00)
    [5/60] 수집: [새상품] 딘토 브로우 픽서 (2026-06-16 00:00:00)
    [6/60] 수집: [새상품] 앰플엔 블레미샷 잡티 크림 30ml (2026-06-16 00:00:00)
    [7/60] 수집: 티비장 중고 처분합니다 (2026-04-17 00:00:00)
    [8/60] 수집: 태그호이어 까레라 칼리버 6 실버 오토매틱 시계 (2026-06-16 00:00:00)
    [9/60] 수집: [마지막가격][품절템] 몽슈레 루나크립 아기침대 풀세트 (2026-06-16 00:00:00)
    [10/60] 수집: 쓰리쿼터 데님 여름 반바지 팬츠L (2026-06-16 00:00:00)
    [11/60] 수집: >>새상품_아이보리 레이스 원피스 S사이즈 (2026-06-16 00:00:00)
    [12/60] 수집: 가스트로플러스 에어리움 40L 오븐에어프라이어 (6/30일 이후 처분) (2026-06-14 00:00:00)
    [13/60] 수집: 사카모토,프리렌,나루토,짱구,미쿠 라부부 처분중 (2026-06-13 00:00:00)
    [14/60] 수집: 보이넥스트도어 탈덕 처분 (2026-03-18 00:00:00)
    [15/60] 수집: 세라 saera구두 225cm 상태굿 처분 (2025-06-16 00:00:00)
    [16/60] 수집: 소파테이블 / 낮은식탁 / 아기책상 판매 (거래 없을 시 폐기처분 

In [1]:
import os
print(os.path.abspath("daangn_crawl_test4.csv"))

C:\Users\user\dataclass_weekday\260617\daangn_crawl_test4.csv


In [ ]:
import time
import csv
import os
import re
from datetime import datetime, timedelta
from urllib.parse import quote

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException

# ---------------------------------------------------------
# 설정
# ---------------------------------------------------------
KEYWORDS = ["빌려"]

# 지역코드 하드코딩 (테스트: 4개 지역)
REGION_CODE_MAP = {
    "송파구": "문정동-6184",
    "강동구": "천호동-6044",
    "서초구": "잠원동-367",
    "관악구": "신림동-355",
    "구로구": "구로동-6064",
    "강서구": "화곡동-6057",
    "마포구": "상암동-237",
    "은평구": "역촌동-200",
    
}

START_DATE = datetime(2025, 6, 1)
END_DATE = datetime(2026, 6, 21, 23, 59, 59)
TODAY = datetime(2026, 6, 16)
OUTPUT_FILE = "daangn_crawl.csv"

SCHEMA = ["post_id", "platform", "keyword", "title", "content_text",
          "view_count", "comment_count", "posted_at", "region"]

SCROLL_TIMES = 30
PAGE_WAIT = 3.0
RESTART_EVERY = 60        # 게시물 N건 처리할 때마다 브라우저 재시작(메모리 누수 방지)


# ---------------------------------------------------------
# 드라이버 생성/재시작
# ---------------------------------------------------------
def make_driver():
    options = Options()
    # options.add_argument("--headless=new")
    options.add_argument("--window-size=1400,1000")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--disable-dev-shm-usage")   # 메모리 부족(Out of Memory) 완화
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-gpu")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                         "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36")
    return webdriver.Chrome(options=options)


driver = make_driver()


# ---------------------------------------------------------
# 유틸
# ---------------------------------------------------------
def to_int(text):
    if not text:
        return None
    nums = re.sub(r"[^\d]", "", text)
    return int(nums) if nums else None


def extract_post_id(url):
    m = re.search(r"-([a-z0-9]{8,})/?$", url)
    if m:
        slug = m.group(1)
        return int.from_bytes(slug.encode(), "big") % (10**18)
    return None


def parse_relative_or_abs(text):
    if not text:
        return None
    text = text.strip()
    for fmt in ("%Y-%m-%dT%H:%M:%S", "%Y-%m-%d %H:%M:%S",
                "%Y.%m.%d", "%Y-%m-%d", "%Y. %m. %d"):
        try:
            return datetime.strptime(text[:len(fmt) + 4].strip(), fmt)
        except ValueError:
            continue
    t = text.replace("끌올", "").strip()
    if "방금" in t or "분 전" in t or "초 전" in t or "시간 전" in t:
        return TODAY
    m = re.search(r"(\d+)\s*일\s*전", t)
    if m:
        return TODAY - timedelta(days=int(m.group(1)))
    m = re.search(r"(\d+)\s*주\s*전", t)
    if m:
        return TODAY - timedelta(weeks=int(m.group(1)))
    m = re.search(r"(\d+)\s*개월\s*전", t)
    if m:
        return TODAY - timedelta(days=int(m.group(1)) * 30)
    m = re.search(r"(\d+)\s*년\s*전", t)
    if m:
        return TODAY - timedelta(days=int(m.group(1)) * 365)
    return None


def in_period(dt):
    return dt is not None and START_DATE <= dt <= END_DATE


def safe_text(by, sel, root=None):
    try:
        return (root or driver).find_element(by, sel).text.strip()
    except Exception:
        return None


def safe_attr(by, sel, attr, root=None):
    try:
        return (root or driver).find_element(by, sel).get_attribute(attr)
    except Exception:
        return None


# ---------------------------------------------------------
# CSV 준비 (헤더 1회 작성, 이후 append)
# ---------------------------------------------------------
def init_csv():
    if not os.path.exists(OUTPUT_FILE):
        with open(OUTPUT_FILE, "w", encoding="utf-8-sig", newline="") as f:
            csv.DictWriter(f, fieldnames=SCHEMA).writeheader()


def append_row(row):
    with open(OUTPUT_FILE, "a", encoding="utf-8-sig", newline="") as f:
        csv.DictWriter(f, fieldnames=SCHEMA).writerow(row)


# ---------------------------------------------------------
# 검색결과 → 게시물 링크 수집
# ---------------------------------------------------------
def collect_links(keyword, region_code):
    url = (f"https://www.daangn.com/kr/buy-sell/s/"
           f"?in={quote(region_code)}&search={quote(keyword)}")
    driver.get(url)
    time.sleep(PAGE_WAIT)

    last_h = driver.execute_script("return document.body.scrollHeight")
    for _ in range(SCROLL_TIMES):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1.8)
        new_h = driver.execute_script("return document.body.scrollHeight")
        if new_h == last_h:
            break
        last_h = new_h

    links, seen = [], set()
    for a in driver.find_elements(By.CSS_SELECTOR, "a[href*='/kr/buy-sell/']"):
        href = a.get_attribute("href")
        if href and re.search(r"/kr/buy-sell/.+-[a-z0-9]{8,}/?$", href):
            if href not in seen:
                seen.add(href)
                links.append(href)
    return links


# ---------------------------------------------------------
# 상세 페이지 크롤링
# ---------------------------------------------------------
def crawl_detail(url, keyword, region):
    driver.get(url)
    time.sleep(PAGE_WAIT)

    post_id = extract_post_id(url)

    title = (safe_text(By.CSS_SELECTOR, "h1")
             or safe_attr(By.CSS_SELECTOR, "meta[property='og:title']", "content"))

    content_text = (safe_text(By.CSS_SELECTOR, "[data-gtm='article_description']")
                    or safe_text(By.CSS_SELECTOR, "article")
                    or safe_attr(By.CSS_SELECTOR, "meta[property='og:description']", "content"))

    dt = (parse_relative_or_abs(safe_attr(By.CSS_SELECTOR, "time", "datetime"))
          or parse_relative_or_abs(safe_text(By.CSS_SELECTOR, "time")))
    if dt is None:
        body = safe_text(By.TAG_NAME, "body") or ""
        m = re.search(r"(끌올\s*)?\d+\s*(일|주|개월|년)\s*전|방금\s*전", body)
        if m:
            dt = parse_relative_or_abs(m.group(0))

    body_text = safe_text(By.TAG_NAME, "body") or ""
    vm = re.search(r"조회\s*([\d,]+)", body_text)
    cm = re.search(r"(?:댓글|관심|채팅)\s*([\d,]+)", body_text)
    view_count = to_int(vm.group(1)) if vm else None
    comment_count = to_int(cm.group(1)) if cm else None

    return {
        "post_id": post_id,
        "platform": "carrot",
        "keyword": keyword,
        "title": title,
        "content_text": content_text,
        "view_count": view_count,
        "comment_count": comment_count,
        "posted_at": dt.strftime("%Y-%m-%d %H:%M:%S") if dt else None,
        "region": region,
    }, dt


# ---------------------------------------------------------
# 메인: 지역 × 키워드
# ---------------------------------------------------------
init_csv()
seen_ids = set()
total = 0
processed = 0      # 재시작 카운터

try:
    for region, code in REGION_CODE_MAP.items():
        print(f"\n##### 지역: {region}  (코드: {code}) #####")

        for kw in KEYWORDS:
            print(f"  ===== 키워드: {kw} =====")
            try:
                links = collect_links(kw, code)
            except Exception as e:
                print(f"  링크 수집 에러: {e} → 드라이버 재시작")
                try:
                    driver.quit()
                except Exception:
                    pass
                driver = make_driver()
                continue
            print(f"  수집된 게시물 링크: {len(links)}")

            for i, url in enumerate(links, 1):
                try:
                    row, dt = crawl_detail(url, kw, region)

                    if row["post_id"] in seen_ids:
                        continue
                    if not in_period(dt):
                        continue

                    seen_ids.add(row["post_id"])
                    append_row(row)          # 즉시 저장
                    total += 1
                    print(f"    [{i}/{len(links)}] 수집: {row['title']} ({row['posted_at']})")

                except (StaleElementReferenceException, TimeoutException) as e:
                    print(f"    [{i}/{len(links)}] {type(e).__name__} skip")
                except Exception as e:
                    print(f"    [{i}/{len(links)}] 에러: {e}")

                processed += 1
                if processed % RESTART_EVERY == 0:
                    print(f"    --- 드라이버 재시작 (누적 {processed}건 처리) ---")
                    try:
                        driver.quit()
                    except Exception:
                        pass
                    driver = make_driver()

                time.sleep(1)

finally:
    try:
        driver.quit()
    except Exception:
        pass

print(f"\n완료: 총 {total}건 저장 → {OUTPUT_FILE}")


In [4]:
import time
import csv
import os
import re
from datetime import datetime, timedelta
from urllib.parse import quote

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException

# ---------------------------------------------------------
# 설정
# ---------------------------------------------------------
KEYWORDS = ["빌려"]

# key=CSV에 기록할 '구' 이름,  value=당근 지역코드(동)
REGION_CODE_MAP = {
    "도봉구": "도봉동-6077",
    "노원구": "상계동-6073",
    "중랑구": "상봉동-6409",
}

START_DATE = datetime(2025, 6, 1)
END_DATE = datetime(2026, 6, 21, 23, 59, 59)
TODAY = datetime(2026, 6, 16)
OUTPUT_FILE = "daangn_community_crawl2.csv"

SCHEMA = ["post_id", "platform", "keyword", "title", "content_text",
          "view_count", "comment_count", "posted_at", "region"]

SCROLL_TIMES = 30
PAGE_WAIT = 3.0
RESTART_EVERY = 60

# 동네생활 탭 검색 URL
SEARCH_URL = "https://www.daangn.com/kr/community/s/?in={code}&search={kw}"


# ---------------------------------------------------------
# 드라이버 생성/재시작
# ---------------------------------------------------------
def make_driver():
    options = Options()
    # options.add_argument("--headless=new")
    options.add_argument("--window-size=1400,1000")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-gpu")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                         "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36")
    return webdriver.Chrome(options=options)


driver = make_driver()


# ---------------------------------------------------------
# 유틸
# ---------------------------------------------------------
def to_int(text):
    if not text:
        return None
    nums = re.sub(r"[^\d]", "", text)
    return int(nums) if nums else None


def extract_post_id(url):
    m = re.search(r"-([a-z0-9]{8,})/?$", url)
    if m:
        slug = m.group(1)
        return int.from_bytes(slug.encode(), "big") % (10**18)
    return None


def parse_datetime_iso(text):
    """동네생활 time 태그의 datetime 속성(ISO) 우선 파싱"""
    if not text:
        return None
    text = text.strip()
    # ISO with timezone: 2025-12-15T02:37:56.085+00:00
    try:
        # 타임존/밀리초 제거 후 파싱
        base = re.sub(r"(\.\d+)?([+-]\d{2}:\d{2}|Z)$", "", text)
        return datetime.strptime(base, "%Y-%m-%dT%H:%M:%S")
    except ValueError:
        pass
    for fmt in ("%Y-%m-%d %H:%M:%S", "%Y.%m.%d", "%Y-%m-%d"):
        try:
            return datetime.strptime(text[:len(fmt) + 4].strip(), fmt)
        except ValueError:
            continue
    return None


def parse_relative(text):
    """'6개월 전', '4일 전' 등 상대시간 → datetime"""
    if not text:
        return None
    t = text.replace("끌올", "").strip()
    if "방금" in t or "분 전" in t or "초 전" in t or "시간 전" in t:
        return TODAY
    m = re.search(r"(\d+)\s*일\s*전", t)
    if m:
        return TODAY - timedelta(days=int(m.group(1)))
    m = re.search(r"(\d+)\s*주\s*전", t)
    if m:
        return TODAY - timedelta(weeks=int(m.group(1)))
    m = re.search(r"(\d+)\s*개월\s*전", t)
    if m:
        return TODAY - timedelta(days=int(m.group(1)) * 30)
    m = re.search(r"(\d+)\s*년\s*전", t)
    if m:
        return TODAY - timedelta(days=int(m.group(1)) * 365)
    return None


def in_period(dt):
    return dt is not None and START_DATE <= dt <= END_DATE


def safe_text(by, sel, root=None):
    try:
        return (root or driver).find_element(by, sel).text.strip()
    except Exception:
        return None


def safe_attr(by, sel, attr, root=None):
    try:
        return (root or driver).find_element(by, sel).get_attribute(attr)
    except Exception:
        return None


# ---------------------------------------------------------
# CSV 준비
# ---------------------------------------------------------
def init_csv():
    if not os.path.exists(OUTPUT_FILE):
        with open(OUTPUT_FILE, "w", encoding="utf-8-sig", newline="") as f:
            csv.DictWriter(f, fieldnames=SCHEMA).writeheader()


def append_row(row):
    with open(OUTPUT_FILE, "a", encoding="utf-8-sig", newline="") as f:
        csv.DictWriter(f, fieldnames=SCHEMA).writerow(row)


# ---------------------------------------------------------
# 검색결과 → 게시물 링크 수집 (community 경로)
# ---------------------------------------------------------
def collect_links(keyword, region_code):
    driver.get(SEARCH_URL.format(code=quote(region_code), kw=quote(keyword)))
    time.sleep(PAGE_WAIT)

    last_h = driver.execute_script("return document.body.scrollHeight")
    for _ in range(SCROLL_TIMES):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1.8)
        new_h = driver.execute_script("return document.body.scrollHeight")
        if new_h == last_h:
            break
        last_h = new_h

    links, seen = [], set()
    for a in driver.find_elements(By.CSS_SELECTOR, "a[href*='/kr/community/']"):
        href = a.get_attribute("href")
        # 상세글: /kr/community/{슬러그}-{8자리이상코드}/
        if href and re.search(r"/kr/community/.+-[a-z0-9]{8,}/?$", href):
            if href not in seen:
                seen.add(href)
                links.append(href)
    return links


# ---------------------------------------------------------
# 상세 페이지 크롤링 (동네생활)
# ---------------------------------------------------------
def crawl_detail(url, keyword, region):
    driver.get(url)
    time.sleep(PAGE_WAIT)

    post_id = extract_post_id(url)

    title = (safe_text(By.CSS_SELECTOR, "h1")
             or safe_attr(By.CSS_SELECTOR, "meta[property='og:title']", "content"))

    content_text = (safe_attr(By.CSS_SELECTOR, "meta[property='og:description']", "content")
                    or safe_text(By.CSS_SELECTOR, "article"))

    # 작성일: 첫 time 태그의 datetime(ISO) 우선, 없으면 본문 상대시간
    dt = parse_datetime_iso(safe_attr(By.CSS_SELECTOR, "time", "datetime"))
    if dt is None:
        dt = parse_relative(safe_text(By.CSS_SELECTOR, "time"))

    # 조회수/댓글수: 본문 텍스트에서 추출
    body_text = safe_text(By.TAG_NAME, "body") or ""
    vm = re.search(r"조회\s*([\d,]+)", body_text)
    # '댓글 수\n3' 형태 → 댓글 수 다음 숫자
    cm = re.search(r"댓글\s*수\s*([\d,]+)", body_text)
    view_count = to_int(vm.group(1)) if vm else None
    comment_count = to_int(cm.group(1)) if cm else None

    return {
        "post_id": post_id,
        "platform": "carrot",
        "keyword": keyword,
        "title": title,
        "content_text": content_text,
        "view_count": view_count,
        "comment_count": comment_count,
        "posted_at": dt.strftime("%Y-%m-%d %H:%M:%S") if dt else None,
        "region": region,
    }, dt


# ---------------------------------------------------------
# 메인: 지역 × 키워드
# ---------------------------------------------------------
init_csv()
seen_ids = set()
total = 0
processed = 0

try:
    for region, code in REGION_CODE_MAP.items():
        print(f"\n##### 지역: {region}  (코드: {code}) #####")

        for kw in KEYWORDS:
            print(f"  ===== 키워드: {kw} =====")
            try:
                links = collect_links(kw, code)
            except Exception as e:
                print(f"  링크 수집 에러: {e} → 드라이버 재시작")
                try:
                    driver.quit()
                except Exception:
                    pass
                driver = make_driver()
                continue
            print(f"  수집된 게시물 링크: {len(links)}")

            for i, url in enumerate(links, 1):
                try:
                    row, dt = crawl_detail(url, kw, region)

                    if row["post_id"] in seen_ids:
                        continue
                    if not in_period(dt):
                        continue

                    seen_ids.add(row["post_id"])
                    append_row(row)
                    total += 1
                    print(f"    [{i}/{len(links)}] 수집: {row['title']} ({row['posted_at']})")

                except (StaleElementReferenceException, TimeoutException) as e:
                    print(f"    [{i}/{len(links)}] {type(e).__name__} skip")
                except Exception as e:
                    print(f"    [{i}/{len(links)}] 에러: {e}")

                processed += 1
                if processed % RESTART_EVERY == 0:
                    print(f"    --- 드라이버 재시작 (누적 {processed}건 처리) ---")
                    try:
                        driver.quit()
                    except Exception:
                        pass
                    driver = make_driver()

                time.sleep(1)

finally:
    try:
        driver.quit()
    except Exception:
        pass

print(f"\n완료: 총 {total}건 저장 → {OUTPUT_FILE}")


##### 지역: 도봉구  (코드: 도봉동-6077) #####
  ===== 키워드: 빌려 =====
  수집된 게시물 링크: 60
    [1/60] 수집: 24ㅔ 여자 낮은 단화 구두 빌려주실 분 찾아요 (2026-06-13 12:26:02)
    [2/60] 수집: 방학동 근처 카포 빌려주실 수 있는 분 계신가요 (2026-05-22 08:39:07)
    [3/60] 수집: 창고 빌려주실분 (2026-05-19 08:48:46)
    [4/60] 수집: 윈도우 10~ 설치 usb 빌려주실 수 있는분 ㅠㅠ (2026-05-02 14:26:45)
    [5/60] 수집: 글루건 하루만 빌려주실 분 있나요? (2026-03-26 10:37:55)
    [6/60] 수집: 전동 드라이버좀 빌려주실분 계신가요?? (2026-06-10 03:39:09)
    [7/60] 수집: 현관문 도어락 다는데 철문 뚫는 홀소 빌려주실분 (2026-05-17 22:47:43)
    [8/60] 수집: 상계주공 4단지 전동드릴 빌려주실 분 있으실까요? (2026-05-30 11:46:32)
    [9/60] 수집: 타일제거 전동드릴 빌려주세요 ㅜㅜ (2026-03-16 09:23:12)
    [10/60] 수집: 함마드릴 2시간만 빌려주실분! (2026-04-25 07:38:44)
    [11/60] 수집: 몽키 스패널 빌려주실 분 (2026-05-16 17:07:42)
    [12/60] 수집: 상계9동 고양이 이동장 빌려주실분!! (2026-05-09 07:31:45)
    [13/60] 수집: 짐 운반 구르마 빌려주실분 (2026-02-28 11:26:45)
    [14/60] 수집: 고양일 이동장 2시간 정도만 빌려주실분 사례금 드립니다 (2026-03-05 03:11:55)
    [15/60] 수집: 여기 15단지인데 내일 교자상 하루만 빌려주실수있는분 계실까요? ㅜㅜ (2026-02-17 05:01:25)
    [16/60] 수집: 고양

In [2]:
import time, re
from urllib.parse import quote
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

options = Options()
options.add_argument("--window-size=1400,1000")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                     "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36")
driver = webdriver.Chrome(options=options)

# 동네생활 검색결과
driver.get("https://www.daangn.com/kr/community/s/?in=송도동-6543&search=빌려")
time.sleep(4)
for _ in range(3):
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(1.5)

print("현재 URL:", driver.current_url)

# 게시물 링크 패턴 확인 (community 경로)
anchors = driver.find_elements(By.TAG_NAME, "a")
hrefs = [a.get_attribute("href") for a in anchors if a.get_attribute("href")]
comm_like = sorted(set(h for h in hrefs if "/community/" in h))
print("\ncommunity 링크 샘플:")
for h in comm_like[:20]:
    print("  ", h)

# 첫 게시물 상세 진입
detail = [h for h in comm_like if re.search(r"/community/.+-[a-z0-9]{8,}/?$", h)]
if detail:
    driver.get(detail[0])
    time.sleep(4)
    print("\n=== 상세페이지 ===")
    print("URL:", driver.current_url)
    print("h1:", [h.text for h in driver.find_elements(By.TAG_NAME, "h1")][:3])
    print("h2:", [h.text for h in driver.find_elements(By.TAG_NAME, "h2")][:3])
    print("time:")
    for t in driver.find_elements(By.TAG_NAME, "time")[:3]:
        print("   text=", t.text, "| datetime=", t.get_attribute("datetime"))
    for prop in ["og:title", "og:description"]:
        v = driver.find_elements(By.CSS_SELECTOR, f"meta[property='{prop}']")
        print(prop, "=", v[0].get_attribute("content")[:150] if v else None)
    print("\n--- 상세 body 앞 1200자 ---")
    print(driver.find_element(By.TAG_NAME, "body").text[:1200])

driver.quit()

현재 URL: https://www.daangn.com/kr/community/s/?in=%EC%86%A1%EB%8F%84%EB%8F%99-6543&search=%EB%B9%8C%EB%A0%A4

community 링크 샘플:
   https://www.daangn.com/kr/community/
   https://www.daangn.com/kr/community/%EA%B0%95%EC%95%84%EC%A7%80-%EC%BC%84%EB%84%AC-%EB%B9%8C%EB%A0%A4%EC%A3%BC%EC%8B%A4-%EC%88%98-%EC%9E%88%EC%9C%BC%EC%8B%A0-%EB%B6%84-%EA%B3%84%EC%8B%A4%EA%B9%8C%EC%9A%94-p31njtibdd3e/
   https://www.daangn.com/kr/community/%EA%B2%B0%ED%98%BC%EC%8B%9D-%ED%95%98%EA%B0%9D%EB%A3%A9-%EB%B9%8C%EB%A0%A4%EC%A3%BC%EC%8B%A4-%EB%B6%84-5rgsuqcy7nbq/
   https://www.daangn.com/kr/community/%EA%B3%B5%ED%95%99%EC%9A%A9%EA%B3%84%EC%82%B0%EA%B8%B0-%EB%B9%8C%EB%A0%A4%EC%A3%BC%EC%8B%A4%EB%B6%84%EA%B3%84%EC%8B%A0%EA%B0%80%EC%9A%94-kphrdergq546/
   https://www.daangn.com/kr/community/%EA%B5%90%EB%B3%B5-%EB%B9%8C%EB%A0%A4%EC%9E%85%EA%B3%A0-%EB%82%A8%EC%9D%98-%ED%95%99%EA%B5%90-%EC%B9%A8%ED%88%AC-bf3taoy32v84/
   https://www.daangn.com/kr/community/%EA%B5%AC%EB%91%90%EC%A2%80-%EB%B9%8C%EB%A0%A4%EC%A3%BC%EC%8

In [ ]:
import time
import csv
import os
import re
from datetime import datetime, timedelta
from urllib.parse import quote

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException

# ---------------------------------------------------------
# 설정
# ---------------------------------------------------------
KEYWORDS = ["빌려"]

REGION_CODE_MAP = {
    "송파구": "문정동-6184",
    "강동구": "천호동-6044",
    "서초구": "잠원동-367",
    "관악구": "신림동-355",
    "구로구": "구로동-6064",
    "강서구": "화곡동-6057",
    "마포구": "상암동-237",
    "은평구": "역촌동-200",
    "도봉구": "도봉동-6077",
    "노원구": "상계동-6073",
    "중랑구": "상봉동-6409"
}

START_DATE = datetime(2025, 6, 1)
END_DATE = datetime(2026, 6, 21, 23, 59, 59)
TODAY = datetime(2026, 6, 16)
OUTPUT_FILE = "daangn_community_KW3.csv"

SCHEMA = ["post_id", "platform", "keyword", "title", "content_text",
          "view_count", "comment_count", "posted_at", "region"]

TARGET_LINKS = 200        # 지역·키워드당 목표 수집 링크 수
MAX_MORE_CLICKS = 60      # '더보기' 최대 클릭 횟수(안전장치)
NO_GROWTH_LIMIT = 4       # 클릭해도 N번 연속 안 늘면 중단
PAGE_WAIT = 3.0
CLICK_WAIT = 2.2          # 더보기 클릭 후 로딩 대기
RESTART_EVERY = 60

# 동네생활 탭 검색 URL
SEARCH_URL = "https://www.daangn.com/kr/community/s/?in={code}&search={kw}"


# ---------------------------------------------------------
# 드라이버 생성/재시작
# ---------------------------------------------------------
def make_driver():
    options = Options()
    # options.add_argument("--headless=new")
    options.add_argument("--window-size=1400,1000")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-gpu")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                         "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36")
    return webdriver.Chrome(options=options)


driver = make_driver()


# ---------------------------------------------------------
# 유틸
# ---------------------------------------------------------
def to_int(text):
    if not text:
        return None
    nums = re.sub(r"[^\d]", "", text)
    return int(nums) if nums else None


def extract_post_id(url):
    m = re.search(r"-([a-z0-9]{8,})/?$", url)
    if m:
        slug = m.group(1)
        return int.from_bytes(slug.encode(), "big") % (10**18)
    return None


def parse_datetime_iso(text):
    """동네생활 time 태그의 datetime 속성(ISO) 우선 파싱"""
    if not text:
        return None
    text = text.strip()
    try:
        base = re.sub(r"(\.\d+)?([+-]\d{2}:\d{2}|Z)$", "", text)
        return datetime.strptime(base, "%Y-%m-%dT%H:%M:%S")
    except ValueError:
        pass
    for fmt in ("%Y-%m-%d %H:%M:%S", "%Y.%m.%d", "%Y-%m-%d"):
        try:
            return datetime.strptime(text[:len(fmt) + 4].strip(), fmt)
        except ValueError:
            continue
    return None


def parse_relative(text):
    """'6개월 전', '4일 전' 등 상대시간 → datetime"""
    if not text:
        return None
    t = text.replace("끌올", "").strip()
    if "방금" in t or "분 전" in t or "초 전" in t or "시간 전" in t:
        return TODAY
    m = re.search(r"(\d+)\s*일\s*전", t)
    if m:
        return TODAY - timedelta(days=int(m.group(1)))
    m = re.search(r"(\d+)\s*주\s*전", t)
    if m:
        return TODAY - timedelta(weeks=int(m.group(1)))
    m = re.search(r"(\d+)\s*개월\s*전", t)
    if m:
        return TODAY - timedelta(days=int(m.group(1)) * 30)
    m = re.search(r"(\d+)\s*년\s*전", t)
    if m:
        return TODAY - timedelta(days=int(m.group(1)) * 365)
    return None


def in_period(dt):
    return dt is not None and START_DATE <= dt <= END_DATE


def safe_text(by, sel, root=None):
    try:
        return (root or driver).find_element(by, sel).text.strip()
    except Exception:
        return None


def safe_attr(by, sel, attr, root=None):
    try:
        return (root or driver).find_element(by, sel).get_attribute(attr)
    except Exception:
        return None


# ---------------------------------------------------------
# CSV 준비
# ---------------------------------------------------------
def init_csv():
    if not os.path.exists(OUTPUT_FILE):
        with open(OUTPUT_FILE, "w", encoding="utf-8-sig", newline="") as f:
            csv.DictWriter(f, fieldnames=SCHEMA).writeheader()


def append_row(row):
    with open(OUTPUT_FILE, "a", encoding="utf-8-sig", newline="") as f:
        csv.DictWriter(f, fieldnames=SCHEMA).writerow(row)


# ---------------------------------------------------------
# '더보기' 버튼 찾기
# ---------------------------------------------------------
def find_more_button():
    # 1) data-gtm 컨테이너 안의 button
    for sel in ["div[data-gtm='search_show_more_articles'] button",
                "div[data-gtm*='show_more'] button"]:
        try:
            return driver.find_element(By.CSS_SELECTOR, sel)
        except Exception:
            continue
    # 2) 텍스트가 '더보기'인 button (fallback)
    for b in driver.find_elements(By.TAG_NAME, "button"):
        try:
            if b.text.strip() == "더보기":
                return b
        except Exception:
            continue
    return None


# ---------------------------------------------------------
# 검색결과 → 게시물 링크 수집 ('더보기' 반복 클릭, community 경로)
# ---------------------------------------------------------
def collect_links(keyword, region_code):
    driver.get(SEARCH_URL.format(code=quote(region_code), kw=quote(keyword)))
    time.sleep(PAGE_WAIT)

    links, seen = [], set()
    no_growth = 0

    def harvest():
        before = len(seen)
        for a in driver.find_elements(By.CSS_SELECTOR, "a[href*='/kr/community/']"):
            href = a.get_attribute("href")
            if href and re.search(r"/kr/community/.+-[a-z0-9]{8,}/?$", href):
                if href not in seen:
                    seen.add(href)
                    links.append(href)
        return len(seen) - before

    harvest()  # 초기 로드분

    for _ in range(MAX_MORE_CLICKS):
        if len(links) >= TARGET_LINKS:
            break

        btn = find_more_button()
        if btn is None:
            break

        try:
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
            time.sleep(0.5)
            driver.execute_script("arguments[0].click();", btn)
        except (StaleElementReferenceException, Exception):
            time.sleep(1)
            btn = find_more_button()
            if btn is None:
                break
            try:
                driver.execute_script("arguments[0].click();", btn)
            except Exception:
                break

        time.sleep(CLICK_WAIT)
        grew = harvest()

        if grew == 0:
            no_growth += 1
            if no_growth >= NO_GROWTH_LIMIT:
                break
        else:
            no_growth = 0

    return links[:TARGET_LINKS]


# ---------------------------------------------------------
# 상세 페이지 크롤링 (동네생활)
# ---------------------------------------------------------
def crawl_detail(url, keyword, region):
    driver.get(url)
    time.sleep(PAGE_WAIT)

    post_id = extract_post_id(url)

    title = (safe_text(By.CSS_SELECTOR, "h1")
             or safe_attr(By.CSS_SELECTOR, "meta[property='og:title']", "content"))

    content_text = (safe_attr(By.CSS_SELECTOR, "meta[property='og:description']", "content")
                    or safe_text(By.CSS_SELECTOR, "article"))

    # 작성일: 첫 time 태그의 datetime(ISO) 우선, 없으면 상대시간
    dt = parse_datetime_iso(safe_attr(By.CSS_SELECTOR, "time", "datetime"))
    if dt is None:
        dt = parse_relative(safe_text(By.CSS_SELECTOR, "time"))

    body_text = safe_text(By.TAG_NAME, "body") or ""
    vm = re.search(r"조회\s*([\d,]+)", body_text)
    cm = re.search(r"댓글\s*수\s*([\d,]+)", body_text)
    view_count = to_int(vm.group(1)) if vm else None
    comment_count = to_int(cm.group(1)) if cm else None

    return {
        "post_id": post_id,
        "platform": "carrot",
        "keyword": keyword,
        "title": title,
        "content_text": content_text,
        "view_count": view_count,
        "comment_count": comment_count,
        "posted_at": dt.strftime("%Y-%m-%d %H:%M:%S") if dt else None,
        "region": region,
    }, dt


# ---------------------------------------------------------
# 메인: 지역 × 키워드
# ---------------------------------------------------------
init_csv()
seen_ids = set()
total = 0
processed = 0

try:
    for region, code in REGION_CODE_MAP.items():
        print(f"\n##### 지역: {region}  (코드: {code}) #####")

        for kw in KEYWORDS:
            print(f"  ===== 키워드: {kw} =====")
            try:
                links = collect_links(kw, code)
            except Exception as e:
                print(f"  링크 수집 에러: {e} → 드라이버 재시작")
                try:
                    driver.quit()
                except Exception:
                    pass
                driver = make_driver()
                continue
            print(f"  수집된 게시물 링크: {len(links)}")

            for i, url in enumerate(links, 1):
                try:
                    row, dt = crawl_detail(url, kw, region)

                    if row["post_id"] in seen_ids:
                        continue
                    if not in_period(dt):
                        continue

                    seen_ids.add(row["post_id"])
                    append_row(row)
                    total += 1
                    print(f"    [{i}/{len(links)}] 수집: {row['title']} ({row['posted_at']})")

                except (StaleElementReferenceException, TimeoutException) as e:
                    print(f"    [{i}/{len(links)}] {type(e).__name__} skip")
                except Exception as e:
                    print(f"    [{i}/{len(links)}] 에러: {e}")

                processed += 1
                if processed % RESTART_EVERY == 0:
                    print(f"    --- 드라이버 재시작 (누적 {processed}건 처리) ---")
                    try:
                        driver.quit()
                    except Exception:
                        pass
                    driver = make_driver()

                time.sleep(1)

finally:
    try:
        driver.quit()
    except Exception:
        pass

print(f"\n완료: 총 {total}건 저장 → {OUTPUT_FILE}")



##### 지역: 송파구  (코드: 문정동-6184) #####
  ===== 키워드: 빌려 =====
  수집된 게시물 링크: 200
    [1/200] 수집: 포커칩좀 빌려주세요! (2026-06-18 11:49:37)
    [2/200] 수집: 집에 드릴있으신분 하루 빌려주실수있나요..? (2026-05-29 12:22:42)
    [3/200] 수집: 혹시 내일 오후 4시쯤 구루마,끌차,카트 빌려주실 수 있는 분… (2026-05-30 17:33:43)
    [4/200] 수집: 끌차, 구루마 빌려주실 분 구합니다ㅠㅠ (2026-05-03 10:16:02)
    [5/200] 수집: 구르마 빌려주실분 ㅠㅠㅠㅜ (2026-06-10 03:01:33)
    [6/200] 수집: 유리창 로봇청소기 빌려주실 분 계신가요 (2026-04-18 03:17:09)
    [7/200] 수집: 마천 거여 ~ 이거 한번만 빌려주실 분 계신가요? 사례해드려요 (2026-05-25 18:10:29)
    [8/200] 수집: 책 빌려드립니다 (2026-03-12 06:07:58)
    [9/200] 수집: 안녕하세요 보드게임 빌려주실 분 구합니더 (2026-03-19 11:46:59)
    [10/200] 수집: 아기등산캐리어 빌려주실분ㅜ (2026-03-15 11:55:43)
    [11/200] 수집: 전동드릴 빌려주실분 계실까요 가락시장역(5000원) (2026-04-11 04:23:36)
    [12/200] 수집: 불판이랑 부르스터 빌려주실분 (2026-04-18 01:28:04)
    [13/200] 수집: 전동드라이버 오늘 한번만 빌려주실분 계실까요ㅠ (2026-02-14 08:57:42)
    [14/200] 수집: 행사용 코스트코 접이식 테이블 빌려주실 분 봐주시면 고맙습니다! (2026-05-15 13:11:33)
    [15/200] 수집: 트립트랩 육각랜치 빌려주실분 계실까요? (2026-04-04 20:10:18)
    

In [ ]:
import time
import csv
import os
import re
from datetime import datetime, timedelta
from urllib.parse import quote

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException

# ---------------------------------------------------------
# 설정
# ---------------------------------------------------------
KEYWORDS = ["1회"]

REGION_CODE_MAP = {
    "송파구": "문정동-6184",
    "강동구": "천호동-6044",
    "서초구": "잠원동-367",
    "관악구": "신림동-355",
    "구로구": "구로동-6064",
    "강서구": "화곡동-6057",
    "마포구": "상암동-237",
    "은평구": "역촌동-200",
    "도봉구": "도봉동-6077",
    "노원구": "상계동-6073",
    "중랑구": "상봉동-6409"
}

START_DATE = datetime(2025, 6, 1)
END_DATE = datetime(2026, 6, 21, 23, 59, 59)
TODAY = datetime(2026, 6, 16)
OUTPUT_FILE = "daangn_crawl_KW3_1.csv"

SCHEMA = ["post_id", "platform", "keyword", "title", "content_text",
          "view_count", "comment_count", "posted_at", "region"]

TARGET_LINKS = 200        # 지역·키워드당 목표 수집 링크 수
MAX_MORE_CLICKS = 60      # '더보기' 최대 클릭 횟수(안전장치)
NO_GROWTH_LIMIT = 4       # 클릭해도 N번 연속 안 늘면 중단
PAGE_WAIT = 3.0
CLICK_WAIT = 2.2          # 더보기 클릭 후 로딩 대기
RESTART_EVERY = 60


# ---------------------------------------------------------
# 드라이버 생성/재시작
# ---------------------------------------------------------
def make_driver():
    options = Options()
    # options.add_argument("--headless=new")
    options.add_argument("--window-size=1400,1000")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-gpu")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                         "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36")
    return webdriver.Chrome(options=options)


driver = make_driver()


# ---------------------------------------------------------
# 유틸
# ---------------------------------------------------------
def to_int(text):
    if not text:
        return None
    nums = re.sub(r"[^\d]", "", text)
    return int(nums) if nums else None


def extract_post_id(url):
    m = re.search(r"-([a-z0-9]{8,})/?$", url)
    if m:
        slug = m.group(1)
        return int.from_bytes(slug.encode(), "big") % (10**18)
    return None


def parse_relative_or_abs(text):
    if not text:
        return None
    text = text.strip()
    for fmt in ("%Y-%m-%dT%H:%M:%S", "%Y-%m-%d %H:%M:%S",
                "%Y.%m.%d", "%Y-%m-%d", "%Y. %m. %d"):
        try:
            return datetime.strptime(text[:len(fmt) + 4].strip(), fmt)
        except ValueError:
            continue
    t = text.replace("끌올", "").strip()
    if "방금" in t or "분 전" in t or "초 전" in t or "시간 전" in t:
        return TODAY
    m = re.search(r"(\d+)\s*일\s*전", t)
    if m:
        return TODAY - timedelta(days=int(m.group(1)))
    m = re.search(r"(\d+)\s*주\s*전", t)
    if m:
        return TODAY - timedelta(weeks=int(m.group(1)))
    m = re.search(r"(\d+)\s*개월\s*전", t)
    if m:
        return TODAY - timedelta(days=int(m.group(1)) * 30)
    m = re.search(r"(\d+)\s*년\s*전", t)
    if m:
        return TODAY - timedelta(days=int(m.group(1)) * 365)
    return None


def in_period(dt):
    return dt is not None and START_DATE <= dt <= END_DATE


def safe_text(by, sel, root=None):
    try:
        return (root or driver).find_element(by, sel).text.strip()
    except Exception:
        return None


def safe_attr(by, sel, attr, root=None):
    try:
        return (root or driver).find_element(by, sel).get_attribute(attr)
    except Exception:
        return None


# ---------------------------------------------------------
# CSV 준비
# ---------------------------------------------------------
def init_csv():
    if not os.path.exists(OUTPUT_FILE):
        with open(OUTPUT_FILE, "w", encoding="utf-8-sig", newline="") as f:
            csv.DictWriter(f, fieldnames=SCHEMA).writeheader()


def append_row(row):
    with open(OUTPUT_FILE, "a", encoding="utf-8-sig", newline="") as f:
        csv.DictWriter(f, fieldnames=SCHEMA).writerow(row)


# ---------------------------------------------------------
# '더보기' 버튼 찾기
# ---------------------------------------------------------
def find_more_button():
    # 1) data-gtm 컨테이너 안의 button (가장 확실)
    try:
        return driver.find_element(
            By.CSS_SELECTOR, "div[data-gtm='search_show_more_articles'] button")
    except Exception:
        pass
    # 2) 텍스트가 '더보기'인 button 탐색 (fallback)
    for b in driver.find_elements(By.TAG_NAME, "button"):
        try:
            if b.text.strip() == "더보기":
                return b
        except Exception:
            continue
    return None


# ---------------------------------------------------------
# 검색결과 → 게시물 링크 수집 ('더보기' 반복 클릭)
# ---------------------------------------------------------
def collect_links(keyword, region_code):
    url = (f"https://www.daangn.com/kr/buy-sell/s/"
           f"?in={quote(region_code)}&search={quote(keyword)}")
    driver.get(url)
    time.sleep(PAGE_WAIT)

    links, seen = [], set()
    no_growth = 0

    def harvest():
        before = len(seen)
        for a in driver.find_elements(By.CSS_SELECTOR, "a[href*='/kr/buy-sell/']"):
            href = a.get_attribute("href")
            if href and re.search(r"/kr/buy-sell/.+-[a-z0-9]{8,}/?$", href):
                if href not in seen:
                    seen.add(href)
                    links.append(href)
        return len(seen) - before

    harvest()  # 초기 로드분

    for _ in range(MAX_MORE_CLICKS):
        if len(links) >= TARGET_LINKS:
            break

        btn = find_more_button()
        if btn is None:
            break  # 더보기 버튼이 없으면 마지막 페이지

        try:
            # 버튼이 보이도록 스크롤 후 클릭 (JS 클릭이 가림 문제에 강함)
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn)
            time.sleep(0.5)
            driver.execute_script("arguments[0].click();", btn)
        except (StaleElementReferenceException, Exception):
            # 버튼이 갱신됐을 수 있으니 다시 한 번 시도
            time.sleep(1)
            btn = find_more_button()
            if btn is None:
                break
            try:
                driver.execute_script("arguments[0].click();", btn)
            except Exception:
                break

        time.sleep(CLICK_WAIT)
        grew = harvest()

        if grew == 0:
            no_growth += 1
            if no_growth >= NO_GROWTH_LIMIT:
                break
        else:
            no_growth = 0

    return links[:TARGET_LINKS]


# ---------------------------------------------------------
# 상세 페이지 크롤링
# ---------------------------------------------------------
def crawl_detail(url, keyword, region):
    driver.get(url)
    time.sleep(PAGE_WAIT)

    post_id = extract_post_id(url)

    title = (safe_text(By.CSS_SELECTOR, "h1")
             or safe_attr(By.CSS_SELECTOR, "meta[property='og:title']", "content"))

    content_text = (safe_text(By.CSS_SELECTOR, "[data-gtm='article_description']")
                    or safe_text(By.CSS_SELECTOR, "article")
                    or safe_attr(By.CSS_SELECTOR, "meta[property='og:description']", "content"))

    dt = (parse_relative_or_abs(safe_attr(By.CSS_SELECTOR, "time", "datetime"))
          or parse_relative_or_abs(safe_text(By.CSS_SELECTOR, "time")))
    if dt is None:
        body = safe_text(By.TAG_NAME, "body") or ""
        m = re.search(r"(끌올\s*)?\d+\s*(일|주|개월|년)\s*전|방금\s*전", body)
        if m:
            dt = parse_relative_or_abs(m.group(0))

    body_text = safe_text(By.TAG_NAME, "body") or ""
    vm = re.search(r"조회\s*([\d,]+)", body_text)
    cm = re.search(r"(?:댓글|관심|채팅)\s*([\d,]+)", body_text)
    view_count = to_int(vm.group(1)) if vm else None
    comment_count = to_int(cm.group(1)) if cm else None

    return {
        "post_id": post_id,
        "platform": "carrot",
        "keyword": keyword,
        "title": title,
        "content_text": content_text,
        "view_count": view_count,
        "comment_count": comment_count,
        "posted_at": dt.strftime("%Y-%m-%d %H:%M:%S") if dt else None,
        "region": region,
    }, dt


# ---------------------------------------------------------
# 메인: 지역 × 키워드
# ---------------------------------------------------------
init_csv()
seen_ids = set()
total = 0
processed = 0

try:
    for region, code in REGION_CODE_MAP.items():
        print(f"\n##### 지역: {region}  (코드: {code}) #####")

        for kw in KEYWORDS:
            print(f"  ===== 키워드: {kw} =====")
            try:
                links = collect_links(kw, code)
            except Exception as e:
                print(f"  링크 수집 에러: {e} → 드라이버 재시작")
                try:
                    driver.quit()
                except Exception:
                    pass
                driver = make_driver()
                continue
            print(f"  수집된 게시물 링크: {len(links)}")

            for i, url in enumerate(links, 1):
                try:
                    row, dt = crawl_detail(url, kw, region)

                    if row["post_id"] in seen_ids:
                        continue
                    if not in_period(dt):
                        continue

                    seen_ids.add(row["post_id"])
                    append_row(row)
                    total += 1
                    print(f"    [{i}/{len(links)}] 수집: {row['title']} ({row['posted_at']})")

                except (StaleElementReferenceException, TimeoutException) as e:
                    print(f"    [{i}/{len(links)}] {type(e).__name__} skip")
                except Exception as e:
                    print(f"    [{i}/{len(links)}] 에러: {e}")

                processed += 1
                if processed % RESTART_EVERY == 0:
                    print(f"    --- 드라이버 재시작 (누적 {processed}건 처리) ---")
                    try:
                        driver.quit()
                    except Exception:
                        pass
                    driver = make_driver()

                time.sleep(1)

finally:
    try:
        driver.quit()
    except Exception:
        pass

print(f"\n완료: 총 {total}건 저장 → {OUTPUT_FILE}")


##### 지역: 송파구  (코드: 문정동-6184) #####
  ===== 키워드: 1회 =====
  수집된 게시물 링크: 200
    [1/200] 수집: 아베다 로즈마리민트 샴푸 1회 사용 (2026-06-16 00:00:00)
    [2/200] 수집: 닥터알파 클렌징폼(1회 사용) (2026-06-16 00:00:00)
    [3/200] 수집: (1회 사용) 제니퍼룸 무선 차량 휴대용 미니 핸디청소기 (2026-06-14 00:00:00)
    [4/200] 수집: 오아 클린이 워터B 구강세정기 화이트(1회사용) (2026-06-13 00:00:00)
    [5/200] 수집: 나스 애프터글로우 리퀴드 블러쉬 원더러스트 (테스트1회) (2026-06-15 00:00:00)
    [6/200] 수집: 발톡스 3D EMS 발 마사지기(1회 사용) (2026-06-07 00:00:00)
    [7/200] 수집: [1회 착용] 소보 키티스 스틸레토 슬링백 5cm (2026-06-12 00:00:00)
    [8/200] 수집: 자라 화이트 단화 38 (시착 1회) (2026-06-12 00:00:00)
    [9/200] 수집: 반팔니트 크림 시착1회 (원가2.0) (2026-06-03 00:00:00)
    [10/200] 수집: 에브리띵모던 휴양지 비치 롱원피스 1회착용 (2026-06-16 00:00:00)
    [11/200] 수집: [1회시착] 레터프롬문 에딘 오프숄더 스웻셔츠 (차콜) (2026-06-10 00:00:00)
    [12/200] 수집: 오픈와이와이 네이비 볼캡(1회 착용) (2026-06-11 00:00:00)
    [13/200] 수집: (1회 착용) 무용 발레 빅사이즈 토트백 블랙 (2026-06-11 00:00:00)
